In [2]:
!pip3 install -U ucimlrepo 

In [64]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency ,ttest_ind
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [4]:
from ucimlrepo import fetch_ucirepo, list_available_datasets

#import adult dataset
adult_income = fetch_ucirepo(id=2)

# access data
X = adult_income.data.features
y = adult_income.data.targets

adult_df = pd.concat([X,y],axis=1)

# first ten rows in the data
adult_df.head(10)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
5,37,Private,284582,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0,0,40,United-States,<=50K
6,49,Private,160187,9th,5,Married-spouse-absent,Other-service,Not-in-family,Black,Female,0,0,16,Jamaica,<=50K
7,52,Self-emp-not-inc,209642,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,45,United-States,>50K
8,31,Private,45781,Masters,14,Never-married,Prof-specialty,Not-in-family,White,Female,14084,0,50,United-States,>50K
9,42,Private,159449,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,5178,0,40,United-States,>50K


In [5]:
## Checking shape of data
adult_df.shape

(48842, 15)

##### There are 48,842 rows and 15 columns

In [7]:
adult_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       47879 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education-num   48842 non-null  int64 
 5   marital-status  48842 non-null  object
 6   occupation      47876 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital-gain    48842 non-null  int64 
 11  capital-loss    48842 non-null  int64 
 12  hours-per-week  48842 non-null  int64 
 13  native-country  48568 non-null  object
 14  income          48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


In [8]:
adult_df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,48842.0,38.643585,13.710510,17.0,28.0,37.0,48.0,90.0
fnlwgt,48842.0,189664.134597,105604.025423,12285.0,117550.5,178144.5,237642.0,1490400.0
education-num,48842.0,10.078089,2.570973,1.0,9.0,10.0,12.0,16.0
capital-gain,48842.0,1079.067626,7452.019058,0.0,0.0,0.0,0.0,99999.0
capital-loss,48842.0,87.502314,403.004552,0.0,0.0,0.0,0.0,4356.0
hours-per-week,48842.0,40.422382,12.391444,1.0,40.0,40.0,45.0,99.0


In [9]:
adult_df.describe(include='object').T

,count,unique,top,freq
workclass,47879,9,Private,33906
education,48842,16,HS-grad,15784
marital-status,48842,7,Married-civ-spouse,22379
occupation,47876,15,Prof-specialty,6172
relationship,48842,6,Husband,19716
race,48842,5,White,41762
sex,48842,2,Male,32650
native-country,48568,42,United-States,43832
income,48842,4,<=50K,24720


In [10]:
## checking null
adult_df.isnull().sum()

age                 0
workclass         963
fnlwgt              0
education           0
education-num       0
marital-status      0
occupation        966
relationship        0
race                0
sex                 0
capital-gain        0
capital-loss        0
hours-per-week      0
native-country    274
income              0
dtype: int64

In [11]:
## Checking duplicates
adult_df.duplicated().sum()

np.int64(29)

In [12]:
## unique data values
adult_df.nunique()

age                  74
workclass             9
fnlwgt            28523
education            16
education-num        16
marital-status        7
occupation           15
relationship          6
race                  5
sex                   2
capital-gain        123
capital-loss         99
hours-per-week       96
native-country       42
income                4
dtype: int64

In [13]:
adult_df['race'].value_counts()

race
White                 41762
Black                  4685
Asian-Pac-Islander     1519
Amer-Indian-Eskimo      470
Other                   406
Name: count, dtype: int64

In [14]:
adult_df['sex'].value_counts()

sex
Male      32650
Female    16192
Name: count, dtype: int64

In [15]:
adult_df['income'].value_counts()

income
<=50K     24720
<=50K.    12435
>50K       7841
>50K.      3846
Name: count, dtype: int64

In [16]:
## replacing <=50k. with <=50k and >50K. with >50K
adult_df['income'] = adult_df['income'].replace({
                     "<=50K.":"<=50K",
                     ">50K.":">50K"})

In [17]:
adult_df['income'].value_counts()

income
<=50K    37155
>50K     11687
Name: count, dtype: int64

In [18]:
adult_df['workclass'].value_counts()

workclass
Private             33906
Self-emp-not-inc     3862
Local-gov            3136
State-gov            1981
?                    1836
Self-emp-inc         1695
Federal-gov          1432
Without-pay            21
Never-worked           10
Name: count, dtype: int64

In [19]:
adult_df['workclass'] = adult_df['workclass'].replace('?','Unknown')

In [20]:
adult_df['occupation'].value_counts()

occupation
Prof-specialty       6172
Craft-repair         6112
Exec-managerial      6086
Adm-clerical         5611
Sales                5504
Other-service        4923
Machine-op-inspct    3022
Transport-moving     2355
Handlers-cleaners    2072
?                    1843
Farming-fishing      1490
Tech-support         1446
Protective-serv       983
Priv-house-serv       242
Armed-Forces           15
Name: count, dtype: int64

In [21]:
adult_df['occupation'] = adult_df['occupation'].replace('?','Unknown')

In [22]:
adult_df['native-country'].value_counts()

native-country
United-States                 43832
Mexico                          951
?                               583
Philippines                     295
Germany                         206
Puerto-Rico                     184
Canada                          182
El-Salvador                     155
India                           151
Cuba                            138
England                         127
China                           122
South                           115
Jamaica                         106
Italy                           105
Dominican-Republic              103
Japan                            92
Guatemala                        88
Poland                           87
Vietnam                          86
Columbia                         85
Haiti                            75
Portugal                         67
Taiwan                           65
Iran                             59
Greece                           49
Nicaragua                        49
Peru         

In [23]:
adult_df['native-country'] = adult_df['native-country'].replace('?','Unknown')

## Hypothesis Testing


#### Stating Null and Alternative Hypothesis 


## Null Hypothesis

*Null Hypothesis is a statement that there is no difference or no effect or no relationship ,It states the default status .We denote this with Ho. We assume it is true until evidence suggests otherwise . We will use statistical test to try to reject or fail to reject the null hypothesis* .

## Alternative Hypothesis

*Alternative hypothesis is a statement that there is a difference or an effect or a relationship between variables . Contradicts the null hypothesis .Denoted by Ha . Accepted if null hypothesis is rejected* .

## Hypothesis 1: Education vs Income

* Null Hypothesis (Ho): There is no effect of Education on Income.
* Alternative Hypothesis (Ha): There is an effect .
* We can use chi square test because these are categorical values .

In [75]:
contingency = pd.crosstab(adult_df['education'],adult_df['income'])
chi2,p,dof,expected = chi2_contingency(contingency)

print("Chi-square",chi2)
if (p<0.05):
    print("Reject the null hypothesis")
else:
    print("fail to reject the null hypothesis")
    

Chi-square 6537.972961360963
Reject the null hypothesis


Since p value is less than level of significance (0.05) ,i.e. the probability of this happening is extremely unlikely under null hypothesis is true ,Therefore We reject the null hypothesis . Education has an effect on income.

## Hypothesis 2: Workclass vs Income

* Null Hypothesis (Ho): There is no effect of workclass on Income.
* Alternative Hypothesis (Ha): There is an effect .
* We can use chi square test because these are categorical values .

In [87]:
contingency2 = pd.crosstab(adult_df['workclass'],adult_df['income'])
chi2,p,dof,expected = chi2_contingency(contingency2)

print("Chi-square",chi2)
if (p<0.05):
    print("Reject the null hypothesis")
else:
    print("fail to reject the null hypothesis")
    

Chi-square 1457.3600428495567
Reject the null hypothesis


Since p value is less than level of significance (0.05) ,i.e. the probability of this happening is extremely unlikely under null hypothesis is true ,Therefore We reject the null hypothesis . Workclass has an effect on income.

## Hypothesis 3: hours-per-week vs Income

* Null Hypothesis (Ho): Average Hours per week (<=50k income group) = Average Hours per week (>50k income group)
* Alternative Hypothesis (Ha):Average Hours per week (<=50k income group) not equal Average Hours per week (>=50k income group)
* We can uset independent t-test because hours-per-week is numerical value.

In [119]:
high_income_work_hours = adult_df[adult_df['income']== '>50K']['hours-per-week']
low_income_work_hours = adult_df[adult_df['income']== '<=50K']['hours-per-week']

t_stat,p = ttest_ind(high_income_work_hours,low_income_work_hours,equal_var=False)
print("t-stat:", t_stat)
if (p<0.05):
    print("Reject the null hypothesis")
else:
    print("fail to reject the null hypothesis")

t-stat: 54.662247230930255
Reject the null hypothesis


Since p value is less than level of significance (0.05) ,i.e. the probability of this happening is extremely unlikely under null hypothesis is true ,Therefore We reject the null hypothesis . Hours-per-week different for different income groups.